In [6]:
import os
import duckdb

# 1. Ensure the directory exists so DuckDB can create the file
db_dir = 'data'
if not os.path.exists(db_dir):
    os.makedirs(db_dir)

# 2. Initialize the connection locally within your current capsule
# This creates 'spatial_engine.db' inside your 'data/' folder
con = duckdb.connect('data/spatial_engine.db')

# 3. Load the spatial extension
con.execute("INSTALL spatial;")
con.execute("LOAD spatial;")

print("Connection successful and Spatial extension loaded.")

Connection successful and Spatial extension loaded.


In [9]:
import os
if os.path.exists('data/spatial_engine.db'):
    os.remove('data/spatial_engine.db')
    print("Old storage version file removed.")

Old storage version file removed.


In [10]:
import duckdb
# This will now create a NEW file using the 1.5.0 storage format
con = duckdb.connect('data/spatial_engine.db')
con.execute("INSTALL spatial; LOAD spatial;")

In [15]:
import sys
import os

# 1. Add the root to sys.path so the notebook can find the 'shared' folder
root_path = os.path.abspath(os.path.join(os.getcwd(), "../../"))
if root_path not in sys.path:
    sys.path.append(root_path)

# 2. Import and run your shared utility
from shared.utils.db.init_warehouse import initialize_engine
initialize_engine()

# 3. Quick verification check
import duckdb
db_path = os.path.join(root_path, "data", "triangle_engine.db")
con = duckdb.connect(db_path)
print("\nActive Schemas in Engine:")
print(con.execute("SELECT schema_name FROM information_schema.schemata WHERE schema_name IN ('bronze', 'silver', 'gold')").df())
con.close()

--- Initializing Global Medallion Schemas at /workspaces/triangle-spatial-data-engine/data/triangle_engine.db ---
Success! Infrastructure is ready.

Active Schemas in Engine:
  schema_name
0      bronze
1        gold
2      silver


In [17]:
import duckdb

con = duckdb.connect(db_path)
con.execute("INSTALL spatial; LOAD spatial;")

# 1. Load Bus Stops (casting geometry to a compatible GEOMETRY type)
con.execute(f"""
    CREATE OR REPLACE TABLE silver.bus_stops AS 
    SELECT * EXCLUDE (geometry), 
    geometry::GEOMETRY AS geometry 
    FROM read_parquet('{bus_stops_path}');
""")

# 2. Load Grocery Stores
con.execute(f"""
    CREATE OR REPLACE TABLE silver.grocery_stores AS 
    SELECT * EXCLUDE (geometry), 
    geometry::GEOMETRY AS geometry 
    FROM read_parquet('{grocery_path}');
""")

print("Silver tables created with compatible spatial types.")
print(con.execute("SELECT table_name, column_name, data_type FROM information_schema.columns WHERE table_schema = 'silver'").df())
con.close()

Silver tables created with compatible spatial types.
        table_name          column_name data_type
0        bus_stops             geometry  GEOMETRY
1   grocery_stores                 type   VARCHAR
2   grocery_stores                   id    BIGINT
3   grocery_stores                  lat    DOUBLE
4   grocery_stores                  lon    DOUBLE
..             ...                  ...       ...
68  grocery_stores         diet:seafood   VARCHAR
69  grocery_stores           diet:vegan   VARCHAR
70  grocery_stores      diet:vegetarian   VARCHAR
71  grocery_stores  payment:contactless   VARCHAR
72  grocery_stores             geometry  GEOMETRY

[73 rows x 3 columns]
